# Extracción del contenido web de las empresas del estudio

En este notebook se construye un corpus textual que se utilizará posteriormente para analizar la similitud semántica entre el contenido disponible de cada empresa y las respuestas generadas por los modelos de lenguaje (ChatGPT y Gemini).

Para ello, se recopila información textual directamente desde las páginas web oficiales de las empresas seleccionadas, con el objetivo de aproximar el tipo de contenido accesible públicamente en internet.

Las 28 empresas del estudio se dividen en dos grupos en función de su aparición en las respuestas generadas por los modelos:

- **Grupo A:** empresas mencionadas más de una vez en el conjunto de respuestas analizadas. (14 empresas)
- **Grupo B:** empresas del mismo sector que no han sido mencionadas en ninguna de las respuestas (14 empresas)

## Instalación de librerías
Se instalan todas las librerías una sola vez para evitar interrupciones en el proceso. La librería *requests* hace las peticiones HTTP a las webs de las empresas. *BeautifulSoup* analiza el HTML y extrae el texto visible sin todo el ruido (menús, etc...) *lxml* es el parser que usa BeautifulSoup para leer y entender el HTML y XML. Por último pandas organiza los resultados en una tabla y los guarda como CSV.

In [1]:
!pip install requests beautifulsoup4 lxml pandas -q

print("Librerías instaladas correctamente.")

Librerías instaladas correctamente.


## Importaciones

En este apartado se importan todas las librerías necesarias en un único bloque, con el objetivo de mantener el código organizado y evitar repeticiones a lo largo del notebook.

Además, se define la variable **HEADERS**, que se utiliza en las peticiones HTTP. Este paso es importante porque muchas páginas web bloquean o limitan las solicitudes que no identifican como procedentes de un navegador real.

También se incluye un *user-agent* que simula un navegador (en este caso, Chrome) junto con el idioma en español, lo que permite que las páginas respondan de forma más consistente y evita posibles errores durante la extracción del contenido.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os

# Cabecera para simular un navegador real y evitar bloqueos
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print("Importaciones completadas.")

Importaciones completadas.


## 3. Definición de las empresas

En este apartado se define el diccionario con las 28 cadenas incluidas en el estudio, junto con las URLs que se utilizarán para extraer el contenido de cada una de ellas.

La selección de estas páginas no es aleatoria, ya que se busca recoger el contenido que mejor representa a cada empresa. Por ello, se priorizan secciones como la página principal, *quiénes somos* o aquellas donde se describe qué ofrece la empresa.

Este enfoque permite obtener un texto más representativo de cada compañía, lo que oportuno para el posterior análisis de similitud semántica con las respuestas generadas por los modelos de lenguaje.

In [3]:
empresas = {
    # ─── GRUPO A: Mencionadas ───────────────────────────────────────────────
    '100_montaditos': {
        'nombre': '100 Montaditos',
        'grupo': 'A',
        'urls': [
            'https://www.100montaditos.com',
            'https://spain.100montaditos.com/es/nuestra-historia',
        ]
    },
    'goiko': {
        'nombre': 'Goiko',
        'grupo': 'A',
        'urls': [
            'https://www.goiko.com',
            'https://www.goiko.com/es/conocenos/nuestra-historia/',
        ]
    },
    'la_tagliatella': {
        'nombre': 'La Tagliatella',
        'grupo': 'A',
        'urls': [
            'https://www.latagliatella.es',
        ]
    },
    'telepizza': {
        'nombre': 'Telepizza',
        'grupo': 'A',
        'urls': [
            'https://www.telepizza.es',
        ]
    },
    'the_good_burger': {
        'nombre': 'The Good Burger',
        'grupo': 'A',
        'urls': [
            'https://www.thegoodburger.com',
        ]
    },
    'vips': {
        'nombre': 'VIPS',
        'grupo': 'A',
        'urls': [
            'https://www.vips.es',
        ]
    },
    'fosters_hollywood': {
        'nombre': "Foster's Hollywood",
        'grupo': 'A',
        'urls': [
            'https://www.fostershollywood.es',
        ]
    },
    'rodilla': {
        'nombre': 'Rodilla',
        'grupo': 'A',
        'urls': [
            'https://www.rodilla.es',
        ]
    },
    'kfc': {
        'nombre': 'KFC',
        'grupo': 'A',
        'urls': [
            'https://www.kfc.es',
        ]
    },
    'dominos': {
        'nombre': "Domino's Pizza",
        'grupo': 'A',
        'urls': [
            'https://www.dominospizza.es',
        ]
    },

    'ginos': {
        'nombre': 'Ginos',
        'grupo': 'A',
        'urls': [
            'https://www.ginos.es',
        ]
    },
    'lizarran': {
        'nombre': 'Lizarran',
        'grupo': 'A',
        'urls': [
            'https://www.lizarran.es',
        ]
    },
    'popeyes': {
        'nombre': 'Popeyes',
        'grupo': 'A',
        'urls': [
            'https://www.popeyes.es',
        ]
    },
    'taco_bell': {
        'nombre': 'Taco Bell',
        'grupo': 'A',
        'urls': [
            'https://www.tacobell.es',
        ]
    },
    # ─── GRUPO B: No mencionadas (grupo control) ────────────────────────────
    'muerde_la_pasta': {
        'nombre': 'Muerde la Pasta',
        'grupo': 'B',
        'urls': [
            'https://muerdelapasta.com',
        ]
    },
    'la_mafia': {
        'nombre': 'La Mafia se sienta a la mesa',
        'grupo': 'B',
        'urls': [
            'https://lamafia.es',
        ]
    },
    'papizza': {
        'nombre': 'Papizza',
        'grupo': 'B',
        'urls': [
            'https://www.papizza.es',
        ]
    },
    'ribs': {
        'nombre': 'Ribs',
        'grupo': 'B',
        'urls': [
            'https://www.ribs.es',
        ]
    },
    'pomodoro': {
        'nombre': 'Pomodoro',
        'grupo': 'B',
        'urls': [
            'https://pomodoro.es',
        ]
    },
    'vicio': {
        'nombre': 'Vicio',
        'grupo': 'B',
        'urls': [
            'https://vicio.com',
        ]
    },
    'miss_sushi': {
        'nombre': 'Miss Sushi',
        'grupo': 'B',
        'urls': [
            'https://misssushi.es',
        ]
    },
   'sibuya': {
        'nombre': 'Sibuya',
        'grupo': 'B',
        'urls': [
            'https://www.sibuya.es',
        ]
    },
    'pizza_hut': {
        'nombre': 'Pizza Hut',
        'grupo': 'B',
        'urls': [
            'https://www.pizzahut.es',
        ]
    },
'papa_johns': {
        'nombre': "Papa John's",
        'grupo': 'B',
        'urls': [
            'https://www.papajohns.es',
            'https://www.papajohns.es/conocenos/',
        ]
    },
    'cantina_mariachi': {
        'nombre': 'Cantina Mariachi',
        'grupo': 'B',
        'urls': [
            'https://www.cantinamariachi.com/',
        ]
    },
    'o mamma_mia': {
        'nombre': 'O Mamma Mia',
        'grupo': 'B',
        'urls': [
            'https://www.omammamia.com',
        ]
    },
    'padthaiwok': {
        'nombre': 'Padthaiwok',
        'grupo': 'B',
        'urls': [
            'https://www.padthaiwok.com',
        ]
    },
'buga_ramen': {
        'nombre': 'Buga Ramen',
        'grupo': 'B',
        'urls': [
            'https://www.buga-ramen.com',
        ]
    },
}

print(f"  Total empresas:             {len(empresas)}")
print(f"  Grupo A (mencionadas):      {sum(1 for e in empresas.values() if e['grupo'] == 'A')}")
print(f"  Grupo B (no mencionadas):   {sum(1 for e in empresas.values() if e['grupo'] == 'B')}")

  Total empresas:             28
  Grupo A (mencionadas):      14
  Grupo B (no mencionadas):   14


Para cada empresa se han definido entre una y dos URLs. La página principal recoge el contenido general de la marca (propuesta de valor, productos o experiencia), mientras que las páginas secundarias, como "quiénes somos", "nuestra historia" o "sobre nosotros", aportan información más descriptiva sobre la empresa.

No todas las empresas cuentan con una segunda URL, ya que este tipo de contenido no siempre está disponible de forma accesible. En algunos casos la página no existe, redirige a la principal o no aporta contenido relevante. Por este motivo, en esas situaciones se trabaja únicamente con la página principal, siempre que proporcione una cantidad de texto suficiente para representar a la empresa.

## 4. Limpieza y extracción de texto

En este apartado se definen las funciones que se utilizarán a lo largo del notebook para llevar a cabo la extracción y el procesamiento del contenido web.

Por un lado, la función `limpiar_texto(texto)` se encarga de depurar el texto extraído. Tras la extracción, es habitual que el contenido incluya espacios innecesarios, saltos de línea o elementos que no aportan información relevante, por lo que esta función permite obtener un texto más limpio y preparado para su análisis.

Por otro lado, la función `extraer_texto_url(url)` realiza la petición HTTP a la página web y procesa su contenido. Para ello, utiliza BeautifulSoup para analizar el HTML y eliminar aquellos elementos que no contienen información semántica relevante, como `script`, `style` o menús de navegación.

Además, en caso de que la petición no se complete correctamente, la función devuelve un diccionario con un estado distinto de "ok", lo que permite identificar posibles errores durante el proceso y controlarlos en el resumen final.

In [4]:
def limpiar_texto(texto):
    """Limpia el texto extraído eliminando espacios innecesarios,
    saltos de línea repetidos y fragmentos demasiado cortos."""
    texto = re.sub(r'\n\s*\n+', '\n\n', texto)
    texto = '\n'.join(linea.strip() for linea in texto.splitlines())
    lineas = [l for l in texto.splitlines() if len(l.strip()) > 3]
    return '\n'.join(lineas).strip()


def extraer_texto_url(url, timeout=15):
    """Realiza la petición a una URL y extrae el texto visible relevante."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, 'lxml')

        # Eliminar etiquetas que no aportan contenido textual relevante
        for tag in soup([
            'script', 'style', 'noscript', 'iframe',
            'nav', 'footer', 'header', 'aside',
            'form', 'button', 'input', 'meta', 'link'
        ]):
            tag.decompose()

        texto = soup.get_text(separator='\n')
        texto = limpiar_texto(texto)

        return {
            'url': url,
            'status': 'ok',
            'texto': texto,
            'num_caracteres': len(texto)
        }

    except requests.exceptions.Timeout:
        return {
            'url': url,
            'status': 'timeout',
            'texto': '',
            'num_caracteres': 0
        }

    except requests.exceptions.HTTPError as e:
        return {
            'url': url,
            'status': f'error_http_{e.response.status_code}',
            'texto': '',
            'num_caracteres': 0
        }

    except Exception as e:
        return {
            'url': url,
            'status': f'error: {str(e)[:50]}',
            'texto': '',
            'num_caracteres': 0
        }


print('Funciones definidas correctamente.')

Funciones definidas correctamente.


## 5. Extracción del texto

En este apartado se recorre el conjunto de empresas y las URLs asociadas a cada una de ellas para extraer el contenido textual disponible. El texto obtenido de cada URL se agrupa posteriormente a nivel de empresa, de forma que cada compañía queda representada por un único bloque de contenido.

En primer lugar, la extracción se realiza mediante peticiones HTTP utilizando la librería requests, lo que permite obtener el contenido HTML de forma rápida y eficiente en la mayoría de los casos.

Sin embargo, algunas páginas web cargan su contenido de forma dinámica mediante JavaScript, por lo que no es posible acceder a la información relevante únicamente con requests. En estos casos, cuando el texto extraído es insuficiente o nulo, se recurre a Selenium como método complementario, permitiendo renderizar la página completa antes de extraer el contenido.

Durante el proceso se introduce una pausa de 2 segundos entre peticiones, con el objetivo de evitar sobrecargar los servidores y reducir el riesgo de bloqueo por comportamiento automatizado.

Finalmente, se muestra un resumen con el estado de cada empresa. Aquellas que presentan una cantidad de texto insuficiente o errores en la extracción se identifican para su posterior revisión, ya que esto podría afectar al análisis de similitud semántica.

In [5]:
resultados = {}

for clave, info in empresas.items():
    print(f"\n[{info['grupo']}] {info['nombre']}")
    textos_empresa = []
    urls_ok = []

    for url in info['urls']:
        r = extraer_texto_url(url)
        print(f"   {r['status']:30s} | {r['num_caracteres']:6d} chars | {url}")

        if r['status'] == 'ok' and r['texto']:
            texto_limpio = r['texto'].strip()

            # evitar duplicados exactos
            if texto_limpio not in textos_empresa:
                textos_empresa.append(texto_limpio)
                urls_ok.append(url)

        time.sleep(2)

    texto_total = '\n\n'.join(textos_empresa)

    resultados[clave] = {
        'nombre': info['nombre'],
        'grupo': info['grupo'],
        'texto': texto_total,
        'num_caracteres': len(texto_total),
        'num_fuentes_ok': len(urls_ok),
        'urls_ok': urls_ok
    }

print('\n\n=== RESUMEN ===')
for clave, datos in resultados.items():
    estado = 'OK' if datos['num_caracteres'] > 200 else 'REVISAR'
    print(
        f"{estado} [{datos['grupo']}] {datos['nombre']:<32} "
        f"{datos['num_caracteres']:6d} chars | "
        f"{datos['num_fuentes_ok']} fuentes"
    )


[A] 100 Montaditos
   ok                             |    368 chars | https://www.100montaditos.com
   ok                             |   2742 chars | https://spain.100montaditos.com/es/nuestra-historia

[A] Goiko
   ok                             |   2887 chars | https://www.goiko.com
   ok                             |   1742 chars | https://www.goiko.com/es/conocenos/nuestra-historia/

[A] La Tagliatella
   ok                             |     14 chars | https://www.latagliatella.es

[A] Telepizza
   ok                             |   1789 chars | https://www.telepizza.es

[A] The Good Burger
   ok                             |   1257 chars | https://www.thegoodburger.com

[A] VIPS
   ok                             |   3597 chars | https://www.vips.es

[A] Foster's Hollywood
   ok                             |   2702 chars | https://www.fostershollywood.es

[A] Rodilla
   ok                             |      7 chars | https://www.rodilla.es

[A] KFC
   ok                          

## 6. Guardado provisional del corpus
Aunque aún no tengo todo el corpus listo, hago un primer guardado para asegurarme de no perder todo el progreso. Más adelante, se actualizará con los datos de las empresas faltantes.

In [ ]:
carpeta = 'textos_webs'
os.makedirs(carpeta, exist_ok=True)

df = pd.DataFrame([
    {
        'clave': clave,
        'nombre': datos['nombre'],
        'grupo': datos['grupo'],
        'num_caracteres': datos['num_caracteres'],
        'texto': datos['texto']
    }
    for clave, datos in resultados.items()
])

df = df.sort_values(by='grupo')

ruta_csv = f'{carpeta}/corpus_empresas.csv'
df.to_csv(ruta_csv, index=False, encoding='utf-8')

print(f'CSV guardado en: {ruta_csv}')
print(f'   Total empresas: {len(df)}')
print(f'   Con texto suficiente (>200 chars): {(df["num_caracteres"] > 200).sum()} / {len(df)}')

## 7. Revisión de la extracción inicial

Tras la extracción inicial mediante requests, la mayoría de las empresas devuelven una cantidad de texto suficiente para construir el corpus. Sin embargo, algunas páginas siguen devolviendo contenido insuficiente o nulo.

En estos casos, la causa más probable es que parte del contenido de la web se cargue dinámicamente mediante JavaScript o que existan restricciones de acceso a través de peticiones HTTP directas. Por este motivo, estas empresas se revisan en un paso posterior mediante Selenium, con el objetivo de completar el corpus y mejorar la representatividad del texto recopilado.

Las empresas pendientes de revisión son:

- **Grupo A:** La Tagliatella, Rodilla, KFC, Domino's Pizza, Lizarran, Popeyes y Taco Bell
- **Grupo B:** Pomodoro, Sibuya, Pizza Hut y Cantina Mariachi

In [10]:
!pip install selenium
!pip install webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

In [11]:
empresas_pendientes = {
    'la_tagliatella': empresas['la_tagliatella'],
    'rodilla': empresas['rodilla'],
    'kfc': empresas['kfc'],
    'dominos': empresas['dominos'],
    'lizarran': empresas['lizarran'],
    'popeyes': empresas['popeyes'],
    'taco_bell': empresas['taco_bell'],
    'pomodoro': empresas['pomodoro'],
    'sibuya': empresas['sibuya'],
    'pizza_hut': empresas['pizza_hut'],
    'cantina_mariachi': empresas['cantina_mariachi'],
    'papa_johns': empresas['papa_johns'],
}

In [12]:
def crear_driver():
    """Configura y lanza Chrome en modo headless para cargar páginas
    cuyo contenido depende de JavaScript."""
    opciones = Options()
    opciones.add_argument('--headless=new')
    opciones.add_argument('--no-sandbox')
    opciones.add_argument('--disable-dev-shm-usage')
    opciones.add_argument('--disable-gpu')
    opciones.add_argument('--window-size=1920,1080')
    opciones.add_argument('--lang=es-ES')
    opciones.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    opciones.add_experimental_option('excludeSwitches', ['enable-logging'])

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opciones)


def extraer_con_selenium(driver, url, espera=6):
    """Carga una URL con Selenium y extrae el texto visible una vez
    renderizado el contenido dinámico de la página."""
    try:
        driver.get(url)
        time.sleep(espera)

        # Intentar cerrar banners de cookies si aparecen
        for selector in [
            "button[id*='accept']", "button[id*='cookie']",
            "button[class*='accept']", "button[class*='cookie']",
            "#onetrust-accept-btn-handler", ".cookie-accept"
        ]:
            try:
                btn = driver.find_element(By.CSS_SELECTOR, selector)
                btn.click()
                time.sleep(1)
                break
            except Exception:
                pass

        texto = driver.find_element(By.TAG_NAME, 'body').text
        texto = limpiar_texto(texto)

        return {
            'url': url,
            'status': 'ok',
            'texto': texto,
            'num_caracteres': len(texto)
        }

    except Exception as e:
        return {
            'url': url,
            'status': f'error: {str(e)[:60]}',
            'texto': '',
            'num_caracteres': 0
        }


print('Funciones de Selenium definidas correctamente.')

Funciones de Selenium definidas correctamente.


Se crea el driver y se reutiliza para todas las empresas pendientes evitando abrir y cerrar chrome para cada petición.

In [13]:
resultados_selenium = {}

print('Iniciando Chrome en segundo plano...')
driver = crear_driver()
print('Chrome listo.\n')

try:
    for clave, info in empresas_pendientes.items():
        print(f"[{info['grupo']}] {info['nombre']}")
        textos_empresa = []

        for url in info['urls']:
            r = extraer_con_selenium(driver, url)
            print(f"   {r['status']:15s} | {r['num_caracteres']:6d} chars | {url}")

            # 🔴 filtro más robusto
            if r['status'] == 'ok' and r['texto'] and r['num_caracteres'] > 100:
                textos_empresa.append(r['texto'])

        texto_total = '\n\n'.join(textos_empresa)

        resultados_selenium[clave] = {
            'nombre': info['nombre'],
            'grupo': info['grupo'],
            'texto': texto_total,
            'num_caracteres': len(texto_total)
        }

        print(f"   → Total: {len(texto_total)} chars\n")

finally:
    driver.quit()
    print('Chrome cerrado.')

print('\n=== RESUMEN SELENIUM ===')
for clave, datos in resultados_selenium.items():
    estado = 'OK' if datos['num_caracteres'] > 200 else 'REVISAR'
    print(f"{estado} [{datos['grupo']}] {datos['nombre']:<32} {datos['num_caracteres']:6d} chars")

Iniciando Chrome en segundo plano...
Chrome listo.

[A] La Tagliatella
   ok              |   3343 chars | https://www.latagliatella.es
   → Total: 3343 chars

[A] Rodilla
   ok              |    856 chars | https://www.rodilla.es
   → Total: 856 chars

[A] KFC
   ok              |    219 chars | https://www.kfc.es
   → Total: 219 chars

[A] Domino's Pizza
   ok              |    898 chars | https://www.dominospizza.es
   → Total: 898 chars

[A] Lizarran
   ok              |     49 chars | https://www.lizarran.es
   → Total: 0 chars

[A] Popeyes
   ok              |   2388 chars | https://www.popeyes.es
   → Total: 2388 chars

[A] Taco Bell
   ok              |    376 chars | https://www.tacobell.es
   → Total: 376 chars

[B] Pomodoro
   ok              |     49 chars | https://pomodoro.es
   → Total: 0 chars

[B] Sibuya
   ok              |    731 chars | https://www.sibuya.es
   → Total: 731 chars

[B] Pizza Hut
   ok              |   1881 chars | https://www.pizzahut.es
   → Total: 

## 8. Recuperación manual

En algunos casos, la extracción automática no permite recuperar una cantidad de texto suficiente, ni mediante peticiones HTTP ni mediante el uso de Selenium. Esto puede deberse a limitaciones en la carga del contenido (por ejemplo, contenido dinámico muy fragmentado) o a restricciones que dificultan el scraping automatizado.

En estas situaciones puntuales, se procede a una recuperación manual del contenido, copiando directamente el texto visible de la web oficial de la empresa. Este procedimiento se aplica de forma excepcional y únicamente cuando los métodos anteriores no han permitido obtener un corpus representativo.

Para garantizar la coherencia del análisis, el contenido recopilado proviene exclusivamente de las páginas oficiales de las empresas, evitando en todo momento el uso de fuentes externas. De este modo, se busca capturar cómo cada empresa se presenta y describe a sí misma en el entorno digital.

In [15]:
texto_manual_pomodoro = """
Pomodoro es un concepto de restauración italiana dentro del segmento “fast casual”, que combina características del fast food y del casual dining. Su propuesta se basa en ofrecer platos como pizza, pasta y opciones con influencia tex-mex a precios competitivos.

Forma parte de Comess Group, una compañía española de restauración organizada con amplia presencia nacional e internacional y más de 350 establecimientos. El grupo desarrolla una estrategia multimarca orientada a ofrecer diferentes opciones de consumo a lo largo de la vida del cliente.

La misión del grupo es ofrecer opciones de restauración variadas que cubran distintas necesidades de consumo, mientras que su visión se centra en consolidarse como líder en restauración organizada a nivel internacional mediante un modelo de franquicias.
"""
texto_manual_lizarran = """
Lizarran es una cadena de restaurantes especializada en la gastronomía española, especialmente en tapas y pintxos, ofreciendo una amplia variedad de opciones para compartir en un ambiente informal y animado. El Pincho es la estrella de Lizarran desde hace más de 30 años. Dispone de servicio de delivery y más de 200 locales franquiciados en España, además de presencia internacional en países como Portugal, México, Chile, China y Marruecos. Forma parte de Comess Group de Restauración y fue fundada en 1988.
"""

texto_manual_cantina_mariachi = """
Cantina Mariachi es una cadena de restauración especializada en cocina mexicana con formato de buffet. Fundada por una familia con tradición en gastronomía, ofrece una reinterpretación de la gastronomía mexicana con ingredientes frescos y recetas familiares. Sus restaurantes son espacios llenos de color donde los clientes pueden disfrutar de la mejor cocina mexicana en un ambiente cálido y cercano. Dispone de servicio de delivery y varios locales repartidos por España.
"""

In [16]:
# ── Partir de los resultados en memoria ──────────────────────────────────────
df_requests = pd.DataFrame([
    {
        'clave': clave,
        'nombre': datos['nombre'],
        'grupo': datos['grupo'],
        'texto': datos['texto'],
        'num_caracteres': datos['num_caracteres']
    }
    for clave, datos in resultados.items()
])

# ── Resultados de Selenium ────────────────────────────────────────────────────
df_selenium = pd.DataFrame([
    {
        'clave': clave,
        'nombre': datos['nombre'],
        'grupo': datos['grupo'],
        'texto': datos['texto'],
        'num_caracteres': datos['num_caracteres']
    }
    for clave, datos in resultados_selenium.items()
])

# ── Textos manuales ───────────────────────────────────────────────────────────
df_manual = pd.DataFrame([
    {
        'clave': 'pomodoro',
        'nombre': 'Pomodoro',
        'grupo': 'B',
        'texto': texto_manual_pomodoro,
        'num_caracteres': len(texto_manual_pomodoro)
    },
    {
        'clave': 'lizarran',
        'nombre': 'Lizarran',
        'grupo': 'A',
        'texto': texto_manual_lizarran,
        'num_caracteres': len(texto_manual_lizarran)
    },
    {
        'clave': 'cantina_mariachi',
        'nombre': 'Cantina Mariachi',
        'grupo': 'B',
        'texto': texto_manual_cantina_mariachi,
        'num_caracteres': len(texto_manual_cantina_mariachi)
    }
])

# ── Fusión ────────────────────────────────────────────────────────────────────
claves_a_reemplazar = set(df_selenium['clave'].tolist() + ['pomodoro', 'lizarran', 'cantina_mariachi'])
df_base = df_requests[~df_requests['clave'].isin(claves_a_reemplazar)]

df_final = pd.concat([df_base, df_selenium, df_manual], ignore_index=True)
df_final = df_final.drop_duplicates(subset=['clave'], keep='last')

# ── Guardar ───────────────────────────────────────────────────────────────────
import os
os.makedirs('textos_webs', exist_ok=True)
df_final.to_csv('textos_webs/corpus_final.csv', index=False, encoding='utf-8')

# ── Resumen ───────────────────────────────────────────────────────────────────
print("Corpus final guardado correctamente.")
print(f"Total empresas: {len(df_final)}\n")
for _, row in df_final.sort_values('grupo').iterrows():
    estado = 'OK' if row['num_caracteres'] > 200 else 'REVISAR'
    print(f"{estado} [{row['grupo']}] {row['nombre']:<32} {row['num_caracteres']:6d} chars")

Corpus final guardado correctamente.
Total empresas: 28

OK [A] 100 Montaditos                     3112 chars
OK [A] KFC                                 219 chars
OK [A] Domino's Pizza                      898 chars
OK [A] Lizarran                            511 chars
OK [A] Popeyes                            2388 chars
OK [A] Taco Bell                           376 chars
OK [A] La Tagliatella                     3343 chars
OK [A] Rodilla                             856 chars
OK [A] Ginos                              2887 chars
OK [A] Foster's Hollywood                 2702 chars
OK [A] VIPS                               3597 chars
OK [A] The Good Burger                    1257 chars
OK [A] Telepizza                          1789 chars
OK [A] Goiko                              4631 chars
OK [B] Pizza Hut                          1881 chars
OK [B] Sibuya                              731 chars
OK [B] Pomodoro                            806 chars
OK [B] Papa John's                        

## 9. Corpus final

Una vez completada la extracción mediante requests, la recuperación complementaria con Selenium y, en los casos puntuales en que ninguno de los dos métodos permitió obtener contenido suficiente, la incorporación manual de texto extraído directamente de las webs oficiales, se obtiene el corpus final del estudio.

El corpus está compuesto por 28 empresas, divididas en dos grupos: 14 empresas mencionadas en las respuestas generadas por los modelos y 14 empresas no mencionadas. Para cada una de ellas se dispone de un bloque textual único, construido a partir del contenido accesible en sus páginas web oficiales.

Este corpus constituye la base para la fase posterior del análisis, en la que se calculará la similitud semántica entre el contenido textual de las empresas y las consultas realizadas a los modelos de lenguaje.

Las tres variables binarias presentan diferencias estadísticamente significativas entre grupos. La más discriminante es Wikidata (p=0.000034): el 100% de las empresas del Grupo A tiene entrada en Wikidata frente a solo 3 del Grupo B. Wikipedia ES también discrimina con fuerza (p=0.000069): 13 de 14 empresas mencionadas por los LLMs tienen artículo en Wikipedia en español, frente a solo 2 del Grupo B. Wikipedia EN, aunque con menor potencia (p=0.021), también resulta significativa.

Estos resultados apuntan a que la presencia en fuentes de conocimiento estructurado —especialmente Wikidata y Wikipedia— es uno de los factores más claramente asociados a la aparición de una empresa en las respuestas de los modelos.